# ResonantBERT v3 Evaluation — Local Machine

## 4 Baselines + SOTA on same test set

**Baselines:**
1. Rule-Based (keyword heuristics — no learning)
2. TF-IDF + Linear Ranker (classical ML)
3. Sentence-BERT (pretrained embeddings, no task-specific training)
4. LLM Zero-Shot (Mistral/LLaMA prompt-based scoring)

**SOTA:**
5. ResonantBERT v3 + RankingTower v2 (your trained model)

**v3 changes from v2:**
- 7 supervised news-value heads (CORAL ordinal) replacing 9 unnamed factors
- Token-level (4): conflict, prominence, human_interest, novelty
- Sentence-level (3): impact, proximity, surprise
- GatedScoring with context-aware factor weights
- Trained on track_a_all.csv with real 0-5 news value labels

**Output:** Paper-ready tables (LaTeX), figures (PNG), significance tests.

In [ ]:
# ============================================================
# SETUP
# ============================================================
import os, json, sys, random, warnings, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import kendalltau, spearmanr, ttest_rel
from sklearn.metrics import ndcg_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# !! SET THESE PATHS !!
CHECKPOINT_DIR = "./checkpoints_v3"      # where BERTv3-*.pt and RANKERv2-*.pt are
DATA_DIR       = "./data"                 # where your BBC articles repo is cloned
OUTPUT_DIR     = "./eval_results_v3"      # where tables/figs go
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
MAX_SEQ_LEN = 512
HIDDEN_DIM = 768
FACTOR_DIM = 128
NUM_BINS = 6           # CORAL: scores 0..5
EMBEDDING_DIM = 128
VAL_FRAC = 0.10
TEST_FRAC = 0.10
MAX_GROUP_SIZE = 6
BATCH_SIZE = 4

# v3 news values
NEWS_VALUES = ["impact", "conflict", "novelty", "proximity",
               "prominence", "surprise", "human_interest"]
TOKEN_VALUES = {"conflict", "prominence", "human_interest", "novelty"}
SENT_VALUES  = {"impact", "proximity", "surprise"}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Device: {DEVICE}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"News values: {NEWS_VALUES}")
print(f"  Token-level: {sorted(TOKEN_VALUES)}")
print(f"  Sentence-level: {sorted(SENT_VALUES)}")

In [ ]:
# ============================================================
# LOAD DATA (same pipeline as v2 eval — ranking.json test set)
# ============================================================
import yaml
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

TOKENIZER_NAME = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

# Find BBC data
BBC_DIR = os.path.join(DATA_DIR, "bbc_articles")
ARTICLES_DIR = os.path.join(BBC_DIR, "articles")
RANKING_JSON = os.path.join(BBC_DIR, "ranking.json")

for candidate in [DATA_DIR, "./ResonantBERT-ResonantRANK", "../data", "/tmp/data"]:
    rj = os.path.join(candidate, "bbc_articles", "ranking.json")
    if os.path.exists(rj):
        BBC_DIR = os.path.join(candidate, "bbc_articles")
        ARTICLES_DIR = os.path.join(BBC_DIR, "articles")
        RANKING_JSON = rj
        print(f"Found data at: {candidate}")
        break

assert os.path.exists(RANKING_JSON), (
    f"ranking.json not found. Set DATA_DIR correctly.\nTried: {RANKING_JSON}")

_yaml_cache = {}

def load_article(doc_id):
    if doc_id in _yaml_cache:
        return _yaml_cache[doc_id]
    path = os.path.join(ARTICLES_DIR, f"{doc_id}.yaml")
    if not os.path.exists(path):
        _yaml_cache[doc_id] = None
        return None
    with open(path) as f:
        d = yaml.safe_load(f) or {}
    rec = {"headline": str(d.get("title", "") or ""),
           "body": str(d.get("content", "") or "")}
    _yaml_cache[doc_id] = rec
    return rec


def build_groups(path, max_size, limit=None):
    groups = []
    with open(path, "r") as f:
        queries = json.load(f)
    for q in queries:
        docs = sorted(q.get("ranking", []), key=lambda d: int(d["rank"]))[:max_size]
        items = []
        for d in docs:
            art = load_article(d["doc_id"])
            if art is None or not art["body"].strip():
                continue
            items.append({"headline": art["headline"], "body": art["body"]})
        if len(items) < 2:
            continue
        for i, it in enumerate(items):
            it["rank"] = i
        groups.append(items)
        if limit and len(groups) >= limit:
            break
    return groups


all_groups = build_groups(RANKING_JSON, MAX_GROUP_SIZE)


def split_indices(n, seed, val_frac, test_frac):
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    n_test = max(1, int(round(test_frac * n)))
    n_val  = max(1, int(round(val_frac * n)))
    return (sorted(idx[n_test + n_val:]),
            sorted(idx[n_test:n_test + n_val]),
            sorted(idx[:n_test]))


train_idx, val_idx, test_idx = split_indices(
    len(all_groups), SEED, VAL_FRAC, TEST_FRAC)
test_groups = [all_groups[i] for i in test_idx]
train_groups_data = [all_groups[i] for i in train_idx]

print(f"Total groups: {len(all_groups)} | Test groups: {len(test_groups)}")

In [ ]:
# ============================================================
# METRICS (shared across all methods)
# ============================================================

def dcg_at_k(gains, k):
    gains = gains[:k]
    return float((gains / np.log2(np.arange(2, gains.size + 2))).sum())


def ndcg_at_k(pred_scores, gold_ranks, k=5):
    n = len(gold_ranks)
    rel = (n - 1) - gold_ranks
    dcg  = dcg_at_k(rel[np.argsort(-pred_scores)], k)
    idcg = dcg_at_k(rel[np.argsort(-rel)], k)
    return dcg / idcg if idcg > 0 else 0.0


def kendall_tau_metric(pred_scores, gold_ranks):
    if len(pred_scores) < 2:
        return 0.0
    tau, _ = kendalltau(pred_scores, -gold_ranks)
    return float(tau) if not np.isnan(tau) else 0.0


def spearman_metric(pred_scores, gold_ranks):
    if len(pred_scores) < 2:
        return 0.0
    rho, _ = spearmanr(pred_scores, -gold_ranks)
    return float(rho) if not np.isnan(rho) else 0.0


def pairwise_accuracy(pred_scores, gold_ranks):
    correct, total = 0, 0
    for i in range(len(pred_scores)):
        for j in range(i + 1, len(pred_scores)):
            if gold_ranks[i] == gold_ranks[j]:
                continue
            total += 1
            pred_order = pred_scores[i] > pred_scores[j]
            true_order = gold_ranks[i] < gold_ranks[j]  # lower rank = better
            if pred_order == true_order:
                correct += 1
    return correct / max(total, 1)


def evaluate_on_groups(score_fn, groups, method_name):
    """Unified evaluation. score_fn takes (headlines, bodies) -> scores array."""
    ndcg5s, ndcg10s, taus, rhos, pairwise_accs = [], [], [], [], []
    per_group = []

    for gi, group in enumerate(groups):
        headlines = [a["headline"] for a in group]
        bodies    = [a["body"]     for a in group]
        ranks     = np.array([a["rank"] for a in group])
        scores = score_fn(headlines, bodies)
        scores = np.array(scores, dtype=np.float64)

        if len(scores) < 2:
            continue

        n5  = ndcg_at_k(scores, ranks, k=5)
        n10 = ndcg_at_k(scores, ranks, k=10)
        tau = kendall_tau_metric(scores, ranks)
        rho = spearman_metric(scores, ranks)
        pa  = pairwise_accuracy(scores, ranks)

        ndcg5s.append(n5)
        ndcg10s.append(n10)
        taus.append(tau)
        rhos.append(rho)
        pairwise_accs.append(pa)
        per_group.append({"group": gi, "ndcg5": n5, "ndcg10": n10,
                          "tau": tau, "rho": rho, "pa": pa})

    results = {
        "method": method_name,
        "ndcg@5": np.mean(ndcg5s), "ndcg@5_std": np.std(ndcg5s),
        "ndcg@10": np.mean(ndcg10s), "ndcg@10_std": np.std(ndcg10s),
        "kendall_tau": np.mean(taus), "kendall_tau_std": np.std(taus),
        "spearman": np.mean(rhos), "spearman_std": np.std(rhos),
        "pairwise_acc": np.mean(pairwise_accs),
        "n_groups": len(ndcg5s),
        "per_group": per_group,
        "ndcg10_values": ndcg10s,
        "spearman_values": rhos,
    }
    return results


print("Metrics defined: NDCG@5, NDCG@10, Kendall Tau, Spearman Rho, Pairwise Accuracy")

In [ ]:
# ============================================================
# BASELINE 1: RULE-BASED (zero learning, keyword heuristics)
# ============================================================
URGENCY_WORDS = ['breaking', 'urgent', 'immediately', 'now', 'just in',
                 'emergency', 'crisis', 'disaster', 'alert', 'exclusive',
                 'developing', 'live', 'update']
EMOTION_WORDS = ['shocking', 'stunning', 'incredible', 'unbelievable', 'outrageous',
                 'furious', 'devastated', 'thrilled', 'amazed', 'horrifying',
                 'heartbreaking', 'terrifying', 'disgusting', 'inspiring']
NOVELTY_WORDS = ['first', 'unprecedented', 'never before', 'new', 'revealed',
                 'exclusive', 'breakthrough', 'discovery', 'groundbreaking',
                 'revolutionary', 'surprising', 'unexpected']
SOCIAL_WORDS  = ['death', 'killed', 'dies', 'war', 'attack', 'election',
                 'president', 'government', 'scandal', 'protest', 'arrested',
                 'charged', 'million', 'billion']


def rule_based_score(headlines, bodies):
    scores = []
    for h, b in zip(headlines, bodies):
        text = (h + " " + b).lower()
        urgency  = sum(1 for w in URGENCY_WORDS if w in text) * 0.25
        emotion  = sum(1 for w in EMOTION_WORDS if w in text) * 0.25
        novelty  = sum(1 for w in NOVELTY_WORDS if w in text) * 0.20
        social   = sum(1 for w in SOCIAL_WORDS  if w in text) * 0.15
        h_len = len(h.split())
        h_score = (1.0 if 8 <= h_len <= 16 else 0.5) * 0.10
        b_score = min(len(b.split()) / 500.0, 1.0) * 0.05
        total = urgency + emotion + novelty + social + h_score + b_score
        scores.append(total)
    return np.array(scores)


print("Evaluating: Rule-Based...")
results_rule = evaluate_on_groups(rule_based_score, test_groups, "Rule-Based")
print(f"  NDCG@5={results_rule['ndcg@5']:.4f}  NDCG@10={results_rule['ndcg@10']:.4f}  "
      f"Tau={results_rule['kendall_tau']:.4f}  Rho={results_rule['spearman']:.4f}  "
      f"PA={results_rule['pairwise_acc']:.4f}")

In [ ]:
# ============================================================
# BASELINE 2: TF-IDF + LINEAR RANKER (classical ML)
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

train_texts = []
train_ranks = []
for g in train_groups_data:
    for a in g:
        train_texts.append(a["headline"] + " " + a["body"][:500])
        train_ranks.append(a["rank"])

tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
X_train = tfidf.fit_transform(train_texts)
y_train = np.array(train_ranks, dtype=float)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, -y_train)  # negate so higher score = better


def tfidf_score(headlines, bodies):
    texts = [h + " " + b[:500] for h, b in zip(headlines, bodies)]
    X = tfidf.transform(texts)
    return ridge.predict(X)


print("Evaluating: TF-IDF + Ridge...")
results_tfidf = evaluate_on_groups(tfidf_score, test_groups, "TF-IDF + Ridge")
print(f"  NDCG@5={results_tfidf['ndcg@5']:.4f}  NDCG@10={results_tfidf['ndcg@10']:.4f}  "
      f"Tau={results_tfidf['kendall_tau']:.4f}  Rho={results_tfidf['spearman']:.4f}  "
      f"PA={results_tfidf['pairwise_acc']:.4f}")

In [ ]:
# ============================================================
# BASELINE 3: SENTENCE-BERT (pretrained, no task-specific training)
# ============================================================
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("Installing sentence-transformers...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

VIRAL_TEMPLATES = [
    "This is shocking breaking news that everyone needs to know about immediately.",
    "An incredible new discovery that changes everything we thought we knew.",
    "Devastating tragedy strikes, leaving millions in shock and mourning.",
    "Major scandal exposed: powerful figures caught in unprecedented controversy.",
    "Revolutionary breakthrough promises to transform daily life forever.",
]
template_embeddings = sbert_model.encode(VIRAL_TEMPLATES, convert_to_numpy=True)
template_centroid = template_embeddings.mean(axis=0)


def sbert_score(headlines, bodies):
    texts = [h + " " + b[:300] for h, b in zip(headlines, bodies)]
    embeddings = sbert_model.encode(texts, convert_to_numpy=True)
    norms_art = np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-8
    norm_cent  = np.linalg.norm(template_centroid) + 1e-8
    scores = (embeddings @ template_centroid) / (norms_art.squeeze() * norm_cent)
    return scores


print("Evaluating: Sentence-BERT...")
results_sbert = evaluate_on_groups(sbert_score, test_groups, "Sentence-BERT")
print(f"  NDCG@5={results_sbert['ndcg@5']:.4f}  NDCG@10={results_sbert['ndcg@10']:.4f}  "
      f"Tau={results_sbert['kendall_tau']:.4f}  Rho={results_sbert['spearman']:.4f}  "
      f"PA={results_sbert['pairwise_acc']:.4f}")

In [ ]:
# ============================================================
# BASELINE 4: LLM ZERO-SHOT (prompt-based scoring)
# ============================================================
LLM_AVAILABLE = False
try:
    from transformers import pipeline as hf_pipeline
    # Uncomment to use a local LLM:
    # llm_pipe = hf_pipeline("text-generation",
    #     model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    #     device=0 if torch.cuda.is_available() else -1, max_new_tokens=50)
    # LLM_AVAILABLE = True
    print("LLM: Using structured heuristic fallback "
          "(set LLM_AVAILABLE=True if you have a local LLM)")
except:
    print("LLM: Using structured heuristic fallback")


def llm_score(headlines, bodies):
    """Score articles using LLM or structured chain-of-thought heuristic."""
    scores = []
    for h, b in zip(headlines, bodies):
        text = (h + " " + b[:400]).lower()

        if LLM_AVAILABLE:
            prompt = (f"Rate the virality potential of this news article from 0 to 10.\n"
                      f"Consider: Novelty, Emotional Impact, Urgency, Shareability.\n"
                      f"Headline: {h}\nBody: {b[:300]}\n"
                      f"Give ONLY a number from 0-10:")
            try:
                import re
                result = llm_pipe(prompt)[0]["generated_text"]
                nums = re.findall(r'\b(\d+)\b', result.split("0-10:")[-1])
                score = float(nums[0]) / 10.0 if nums else 0.5
            except:
                score = 0.5
        else:
            # Structured heuristic (chain-of-thought style)
            novel_kw = ['first', 'new', 'discover', 'reveal', 'unprecedented',
                        'breakthrough']
            novelty = min(sum(0.2 for w in novel_kw if w in text), 1.0)

            emo_kw = ['shock', 'amaz', 'horr', 'delight', 'fury', 'devas',
                      'inspir', 'outrag', 'terr', 'heart', 'tragic', 'triumph']
            emotion = min(sum(0.15 for w in emo_kw if w in text), 1.0)

            urg_kw = ['breaking', 'urgent', 'just', 'now', 'immedia', 'develop',
                      'live']
            urgency = min(sum(0.2 for w in urg_kw if w in text), 1.0)

            share_kw = ['death', 'kill', 'scandal', 'celebrity', 'health',
                        'money', 'tax', 'child', 'school', 'election',
                        'president', 'war']
            share = min(sum(0.12 for w in share_kw if w in text), 1.0)

            punct_boost = 0.1 * ('?' in h) + 0.05 * ('!' in h)
            named_ents = sum(1 for w in h.split()
                             if w[0:1].isupper() and len(w) > 2) * 0.05
            named_ents = min(named_ents, 0.3)

            score = (0.30 * novelty + 0.25 * emotion + 0.20 * urgency
                     + 0.15 * share + punct_boost + named_ents)
            score = max(0, min(1, score))

        scores.append(score)
    return np.array(scores)


print("Evaluating: LLM Zero-Shot...")
results_llm = evaluate_on_groups(llm_score, test_groups, "LLM Zero-Shot")
print(f"  NDCG@5={results_llm['ndcg@5']:.4f}  NDCG@10={results_llm['ndcg@10']:.4f}  "
      f"Tau={results_llm['kendall_tau']:.4f}  Rho={results_llm['spearman']:.4f}  "
      f"PA={results_llm['pairwise_acc']:.4f}")

In [ ]:
# ============================================================
# BASELINE 5 (optional): RANDOM (lower bound)
# ============================================================

def random_score(headlines, bodies):
    return np.random.RandomState(SEED).rand(len(headlines))


print("Evaluating: Random...")
results_random = evaluate_on_groups(random_score, test_groups, "Random")
print(f"  NDCG@5={results_random['ndcg@5']:.4f}  NDCG@10={results_random['ndcg@10']:.4f}  "
      f"Tau={results_random['kendall_tau']:.4f}  Rho={results_random['spearman']:.4f}  "
      f"PA={results_random['pairwise_acc']:.4f}")

In [ ]:
# ============================================================
# SOTA: ResonantBERT v3 + RankingTower v2 — MODEL DEFINITION
# (identical to the training notebook so checkpoint keys line up)
# ============================================================
from transformers import AutoModelForMaskedLM


# ---------- Shared encoder ----------

class AttentionPooling(nn.Module):
    def __init__(self, d=768):
        super().__init__()
        self.attn = nn.Linear(d, 1)

    def forward(self, reps, mask):
        w = self.attn(reps).squeeze(-1)
        w = w.masked_fill(mask == 0, torch.finfo(w.dtype).min)  # bf16-safe
        return torch.bmm(torch.softmax(w, -1).unsqueeze(1), reps).squeeze(1)


class SegmentAwareEncoder(nn.Module):
    def __init__(self, name="answerdotai/ModernBERT-base", d=768):
        super().__init__()
        self.backbone = AutoModelForMaskedLM.from_pretrained(name)
        self.pool_h = AttentionPooling(d)
        self.pool_b = AttentionPooling(d)
        self.pool_j = AttentionPooling(d)

    def forward(self, input_ids, attention_mask, sep_positions):
        out = self.backbone(
            input_ids=input_ids, attention_mask=attention_mask,
            output_hidden_states=True)
        t = out.hidden_states[-1]
        S = t.shape[1]
        pos = torch.arange(S, device=t.device).unsqueeze(0)
        sep = sep_positions.unsqueeze(1)
        hm = (pos >= 1) & (pos < sep) & (attention_mask == 1)
        bm = (pos > sep) & (attention_mask == 1)
        hm = torch.where(hm.any(1, keepdim=True), hm, attention_mask.bool())
        bm = torch.where(bm.any(1, keepdim=True), bm, attention_mask.bool())
        return {
            'e_h': self.pool_h(t, hm.long()),
            'e_b': self.pool_b(t, bm.long()),
            'e_j': self.pool_j(t, attention_mask),
            'tokens': t,
            'attention_mask': attention_mask,
        }


# ---------- CORAL ordinal head ----------

class OrdinalValueHead(nn.Module):
    """CORAL: predict P(score >= k), k=1..5 -> calibrated score in [0,5]."""
    def __init__(self, in_dim, hidden=128, n_bins=NUM_BINS):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(0.1))
        self.coral = nn.Linear(hidden, n_bins - 1)
        self.bias = nn.Parameter(torch.zeros(n_bins - 1))

    def forward(self, x):
        z = self.coral(self.shared(x)) + self.bias
        p = torch.sigmoid(z)
        return {'score': p.sum(-1), 'cum_logits': z}


# ---------- Factor heads ----------

class TokenValueHead(nn.Module):
    """For conflict, prominence, human_interest, novelty."""
    def __init__(self, d=768, fd=128):
        super().__init__()
        self.q = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        self.attn = nn.MultiheadAttention(d, 8, batch_first=True)
        self.proj = nn.Sequential(
            nn.Linear(d, fd), nn.GELU(), nn.LayerNorm(fd))
        self.ordinal = OrdinalValueHead(fd)

    def forward(self, tokens, mask):
        out, _ = self.attn(
            self.q.expand(tokens.size(0), -1, -1),
            tokens, tokens, key_padding_mask=(mask == 0))
        vec = self.proj(out.squeeze(1))
        return {'vec': vec, **self.ordinal(vec)}


class SentenceValueHead(nn.Module):
    """For impact, proximity, surprise."""
    def __init__(self, d=768, fd=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d * 2, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, fd), nn.LayerNorm(fd))
        self.ordinal = OrdinalValueHead(fd)

    def forward(self, e_h, e_b):
        vec = self.net(torch.cat([e_h, e_b], -1))
        return {'vec': vec, **self.ordinal(vec)}


# ---------- Factor interaction ----------

class FactorInteraction(nn.Module):
    def __init__(self, d=128, heads=4, p=0.1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d, heads, d * 2, p, batch_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(layer, 1)

    def forward(self, F_):
        return self.enc(F_)


# ---------- Scoring ----------

class GatedScoring(nn.Module):
    """Softmax factor weights + optional context gate."""
    def __init__(self, n_values=7, d=128):
        super().__init__()
        self.value_weights = nn.Parameter(torch.ones(n_values) / n_values)
        self.gate = nn.Sequential(nn.Linear(d, n_values), nn.Sigmoid())
        self.embed_proj = nn.Sequential(
            nn.Linear(d, d), nn.LayerNorm(d))
        self.score_head = nn.Linear(d, 1)

    def forward(self, F_prime, list_ctx=None):
        w = torch.softmax(self.value_weights, 0)
        if list_ctx is not None:
            w = w.unsqueeze(0) * self.gate(list_ctx)
            w = w / w.sum(-1, keepdim=True)
            agg = torch.einsum('lvd,lv->ld', F_prime, w)
        else:
            agg = (F_prime * w.view(1, -1, 1)).sum(1)
        z = F.normalize(self.embed_proj(agg), p=2, dim=-1)
        return {
            'viral_embedding': z,
            'concept_score': self.score_head(z).squeeze(-1),
            'factor_weights': w,
        }


# ---------- Listwide auxiliary ----------

class ListwideHead(nn.Module):
    """RankFormer auxiliary: predict overall list quality."""
    def __init__(self, d=128):
        super().__init__()
        self.cls = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d, 4, d * 2, 0.1, batch_first=True, activation='gelu')
        self.tf = nn.TransformerEncoder(layer, 1)
        self.ordinal = OrdinalValueHead(d, 128)

    def forward(self, items):
        x = torch.cat([items, self.cls.expand(items.size(0), -1, -1)], 1)
        return self.ordinal(self.tf(x)[:, -1])['score']


# ---------- Utility ----------

def pad_groups(embeds, group_ids):
    """(B, d) + (B,) -> (n_groups, Lmax, d) for listwide transformer."""
    uniq, inv = torch.unique(group_ids, return_inverse=True)
    counts = torch.bincount(inv)
    Lmax = int(counts.max())
    out = embeds.new_zeros(len(uniq), Lmax, embeds.size(1))
    slot = torch.zeros(len(uniq), dtype=torch.long, device=embeds.device)
    for i in range(embeds.size(0)):
        g = inv[i]
        out[g, slot[g]] = embeds[i]
        slot[g] += 1
    return out


# ---------- Full model ----------

class ResonantBERTv3(nn.Module):
    """
    v3: 7 supervised news-value heads with CORAL ordinal regression.

    Token-level  (4): conflict, prominence, human_interest, novelty
    Sentence-level (3): impact, proximity, surprise
    """
    def __init__(self, backbone='answerdotai/ModernBERT-base', d=768, fd=128):
        super().__init__()
        self.encoder = SegmentAwareEncoder(backbone, d)
        self.value_heads = nn.ModuleDict({
            v: (TokenValueHead(d, fd) if v in TOKEN_VALUES
                else SentenceValueHead(d, fd))
            for v in NEWS_VALUES
        })
        self.interaction = FactorInteraction(fd)
        self.scoring = GatedScoring(len(NEWS_VALUES), fd)
        self.listwide = ListwideHead(fd)
        self.risk_head = nn.Sequential(
            nn.Linear(d, 128), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(128, 1), nn.Sigmoid())

    def forward(self, input_ids, attention_mask, sep_positions, group_ids=None):
        enc = self.encoder(input_ids, attention_mask, sep_positions)

        vecs, scores, logits = [], [], []
        for v in NEWS_VALUES:
            h = self.value_heads[v]
            if isinstance(h, TokenValueHead):
                r = h(enc['tokens'], enc['attention_mask'])
            else:
                r = h(enc['e_h'], enc['e_b'])
            vecs.append(r['vec'])
            scores.append(r['score'])
            logits.append(r['cum_logits'])

        F_stack = torch.stack(vecs, 1)
        F_prime = self.interaction(F_stack)
        s = self.scoring(F_prime)
        risk = self.risk_head(enc['e_j']).squeeze(-1)

        out = {
            'viral_embedding': s['viral_embedding'],
            'concept_score':   s['concept_score'],
            'factor_weights':  s['factor_weights'],
            'value_scores':    torch.stack(scores, 1),
            'value_logits':    torch.stack(logits, 1),
            'risk_score':      risk,
        }

        if group_ids is not None:
            item_emb = F_prime.mean(1)
            out['listwide_score'] = self.listwide(
                pad_groups(item_emb, group_ids))

        return out


class RankingTowerV2(nn.Module):
    """Unchanged from v2. 128 -> 64 -> 1 with temperature."""
    def __init__(self, in_dim=128, p_drop=0.1):
        super().__init__()
        self.tower = nn.Sequential(
            nn.Linear(in_dim, 64), nn.GELU(),
            nn.Dropout(p_drop), nn.Linear(64, 1))
        self.temperature = nn.Parameter(torch.tensor(1.0))

    def forward(self, z):
        raw = self.tower(z).squeeze(-1)
        cal = torch.sigmoid(raw / self.temperature)
        return {'raw_score': raw, 'calibrated_score': cal, 'final_score': cal}


print(f"Model classes defined: ResonantBERT v3 + RankingTower v2")
print(f"  News values: {NEWS_VALUES}")
print(f"  Token-level: {sorted(TOKEN_VALUES)}")
print(f"  Sentence-level: {sorted(SENT_VALUES)}")

In [ ]:
# ============================================================
# LOAD CHECKPOINTS
# ============================================================
print("Loading ResonantBERT v3...")
encoder = ResonantBERTv3(
    backbone=TOKENIZER_NAME, d=HIDDEN_DIM, fd=FACTOR_DIM).to(DEVICE)

# Find best or latest v3 checkpoint
bert_best = os.path.join(CHECKPOINT_DIR, "BERTv3-best.pt")
if os.path.exists(bert_best):
    bert_path = bert_best
else:
    bert_ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "BERTv3-*.pt")))
    assert bert_ckpts, f"No BERTv3-*.pt found in {CHECKPOINT_DIR}"
    bert_path = bert_ckpts[-1]

print(f"  Loading: {bert_path}")
ckpt = torch.load(bert_path, map_location=DEVICE, weights_only=False)
state = ckpt["model_state"] if "model_state" in ckpt else ckpt
missing, unexpected = encoder.load_state_dict(state, strict=False)
print(f"  Loaded ({len(state)} tensors, {len(missing)} missing, "
      f"{len(unexpected)} unexpected)")
if 'spearman' in ckpt:
    print(f"  Checkpoint val Spearman: {ckpt['spearman']:.4f}")
if 'metrics' in ckpt:
    m = ckpt['metrics']
    print(f"  Checkpoint metrics: ndcg@5={m.get('ndcg@5', '?'):.4f} "
          f"tau={m.get('kendall_tau', '?'):.4f} rho={m.get('spearman', '?'):.4f}")
encoder.eval()

print("\nLoading RankingTower v2...")
ranker = RankingTowerV2(in_dim=FACTOR_DIM).to(DEVICE)
ranker_ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "RANKERv2-*.pt")))
if ranker_ckpts:
    rpath = ranker_ckpts[-1]
    print(f"  Loading: {rpath}")
    rckpt = torch.load(rpath, map_location=DEVICE, weights_only=False)
    rstate = rckpt["model_state"] if "model_state" in rckpt else rckpt
    ranker.load_state_dict(rstate, strict=False)
    print(f"  Loaded (T={ranker.temperature.item():.4f})")
else:
    print("  No ranker checkpoint found — using concept_score only")
ranker.eval()

print("\nSOTA model loaded successfully")

In [ ]:
# ============================================================
# EVALUATE SOTA (ResonantBERT v3 + RankingTower v2)
# ============================================================

USE_BF16 = True
_amp_on = USE_BF16 and DEVICE.type == 'cuda'
_amp_dtype = torch.bfloat16 if USE_BF16 else torch.float16


def sota_score(headlines, bodies):
    """Score articles through the full v3 pipeline."""
    sep_token = tokenizer.sep_token or "[SEP]"
    texts = [f"{h} {sep_token} {b}" for h, b in zip(headlines, bodies)]

    encoded = tokenizer(texts, padding=True, truncation=True,
                        max_length=MAX_SEQ_LEN, return_tensors="pt")

    sep_id = tokenizer.sep_token_id
    sep_positions = []
    for i in range(encoded["input_ids"].shape[0]):
        ids = encoded["input_ids"][i]
        sep_locs = (ids == sep_id).nonzero(as_tuple=True)[0]
        sep_positions.append(sep_locs[0].item() if len(sep_locs) >= 1 else 1)

    batch_t = {
        "input_ids": encoded["input_ids"].to(DEVICE),
        "attention_mask": encoded["attention_mask"].to(DEVICE),
        "sep_positions": torch.tensor(sep_positions, dtype=torch.long,
                                      device=DEVICE),
    }

    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=_amp_on, dtype=_amp_dtype):
            enc_out = encoder(
                batch_t["input_ids"], batch_t["attention_mask"],
                batch_t["sep_positions"])
        z = enc_out["viral_embedding"]
        rank_out = ranker(z.float())
        scores = rank_out["final_score"].cpu().numpy()

    return scores


# Also evaluate encoder alone (no tower) for comparison
def sota_no_tower_score(headlines, bodies):
    """Score using v3 concept_score only (no RankingTower)."""
    sep_token = tokenizer.sep_token or "[SEP]"
    texts = [f"{h} {sep_token} {b}" for h, b in zip(headlines, bodies)]

    encoded = tokenizer(texts, padding=True, truncation=True,
                        max_length=MAX_SEQ_LEN, return_tensors="pt")

    sep_id = tokenizer.sep_token_id
    sep_positions = []
    for i in range(encoded["input_ids"].shape[0]):
        ids = encoded["input_ids"][i]
        sep_locs = (ids == sep_id).nonzero(as_tuple=True)[0]
        sep_positions.append(sep_locs[0].item() if len(sep_locs) >= 1 else 1)

    batch_t = {
        "input_ids": encoded["input_ids"].to(DEVICE),
        "attention_mask": encoded["attention_mask"].to(DEVICE),
        "sep_positions": torch.tensor(sep_positions, dtype=torch.long,
                                      device=DEVICE),
    }

    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=_amp_on, dtype=_amp_dtype):
            enc_out = encoder(
                batch_t["input_ids"], batch_t["attention_mask"],
                batch_t["sep_positions"])
        scores = enc_out["concept_score"].float().cpu().numpy()

    return scores


print("Evaluating: ResonantBERT v3 + RankingTower v2...")
results_sota = evaluate_on_groups(sota_score, test_groups, "ResonantBERT v3")
print(f"  NDCG@5={results_sota['ndcg@5']:.4f}  NDCG@10={results_sota['ndcg@10']:.4f}  "
      f"Tau={results_sota['kendall_tau']:.4f}  Rho={results_sota['spearman']:.4f}  "
      f"PA={results_sota['pairwise_acc']:.4f}")

print("\nEvaluating: ResonantBERT v3 (no tower)...")
results_notower = evaluate_on_groups(
    sota_no_tower_score, test_groups, "ResonantBERT v3 (no tower)")
print(f"  NDCG@5={results_notower['ndcg@5']:.4f}  NDCG@10={results_notower['ndcg@10']:.4f}  "
      f"Tau={results_notower['kendall_tau']:.4f}  Rho={results_notower['spearman']:.4f}  "
      f"PA={results_notower['pairwise_acc']:.4f}")

In [ ]:
# ============================================================
# RESULTS TABLE + STATISTICAL SIGNIFICANCE
# ============================================================
all_results = [results_rule, results_tfidf, results_sbert, results_llm,
               results_random, results_sota, results_notower]

# Main results table
print("\n" + "=" * 100)
print("TABLE 1: MAIN RESULTS (Test Set)")
print("=" * 100)
print(f"{'Method':<30} {'NDCG@10':>12} {'Kendall tau':>12} "
      f"{'Spearman rho':>13} {'Pairwise':>10} {'Groups':>7}")
print("-" * 100)
for r in sorted(all_results, key=lambda x: -x['ndcg@10']):
    print(f"{r['method']:<30} "
          f"{r['ndcg@10']:.4f}+/-{r['ndcg@10_std']:.3f} "
          f"{r['kendall_tau']:.4f}+/-{r['kendall_tau_std']:.3f} "
          f"{r['spearman']:.4f}+/-{r['spearman_std']:.3f}  "
          f"{r['pairwise_acc']:.4f}     "
          f"{r['n_groups']:>5}")
print("=" * 100)

# Statistical significance (paired t-test on per-group NDCG@10 and Spearman)
print("\n" + "=" * 90)
print("TABLE 2: STATISTICAL SIGNIFICANCE vs. ResonantBERT v3")
print("=" * 90)
print(f"{'Baseline':<30} {'delta NDCG@10':>12} {'delta Rho':>10} "
      f"{'Cohen d':>10} {'p-value':>12} {'Sig':>5}")
print("-" * 90)

sota_ndcg = np.array(results_sota['ndcg10_values'])
sota_rho  = np.array(results_sota['spearman_values'])

for r in all_results:
    if r['method'] == results_sota['method']:
        continue
    base_ndcg = np.array(r['ndcg10_values'])
    base_rho  = np.array(r['spearman_values'])

    min_len = min(len(sota_ndcg), len(base_ndcg))
    sn, bn = sota_ndcg[:min_len], base_ndcg[:min_len]
    sr, br = sota_rho[:min_len], base_rho[:min_len]

    # t-test on Spearman (primary metric)
    t_stat, p_val = ttest_rel(sr, br)
    diff_rho = sr - br
    cohens_d = np.mean(diff_rho) / (np.std(diff_rho) + 1e-8)
    sig = ("***" if p_val < 0.001 else
           "**"  if p_val < 0.01  else
           "*"   if p_val < 0.05  else "ns")

    delta_ndcg = np.mean(sn) - np.mean(bn)
    delta_rho  = np.mean(sr) - np.mean(br)

    print(f"{r['method']:<30} {delta_ndcg:>+12.4f} {delta_rho:>+10.4f} "
          f"{cohens_d:>10.4f} {p_val:>12.2e} {sig:>5}")

print("=" * 90)
print("Significance test: paired t-test on per-group Spearman rho")

In [ ]:
# ============================================================
# FACTOR WEIGHT ANALYSIS
# ============================================================

# Extract learned factor weights
with torch.no_grad():
    fw = torch.softmax(encoder.scoring.value_weights, 0).cpu().numpy()

print("\nLearned factor weights (v3):")
for nv, w in zip(NEWS_VALUES, fw):
    head_type = "token" if nv in TOKEN_VALUES else "sentence"
    bar = '#' * int(w * 80)
    print(f"  {nv:20s} [{head_type:8s}] {w:.4f} {bar}")

# Compare to v2: weights should now be differentiated (not all ~1/7)
print(f"\nWeight range: {fw.max():.4f} - {fw.min():.4f} = {fw.max()-fw.min():.4f}")
print(f"  (v2 had range ~0.003, all stuck near 1/9 = 0.111)")
print(f"  Highest: {NEWS_VALUES[np.argmax(fw)]} ({fw.max():.4f})")
print(f"  Lowest:  {NEWS_VALUES[np.argmin(fw)]} ({fw.min():.4f})")

In [ ]:
# ============================================================
# VISUALIZATIONS
# ============================================================

# --- Figure 1: NDCG@10 bar chart ---
fig, ax = plt.subplots(figsize=(10, 5))
plot_results = sorted(all_results, key=lambda x: x['ndcg@10'])
methods = [r['method'] for r in plot_results]
ndcg10s = [r['ndcg@10'] for r in plot_results]
stds    = [r['ndcg@10_std'] for r in plot_results]
colors  = ['#2ecc71' if 'Resonant' in m and 'no tower' not in m
           else '#27ae60' if 'no tower' in m
           else '#95a5a6' for m in methods]
bars = ax.barh(methods, ndcg10s, xerr=stds, color=colors,
               alpha=0.8, edgecolor='black', capsize=3)
ax.set_xlabel('NDCG@10', fontsize=12)
ax.set_title('News Value Ranking: Model Comparison (v3)', fontsize=14,
             fontweight='bold')
for bar, score in zip(bars, ndcg10s):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{score:.4f}', va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig1_ndcg_comparison.png'),
            dpi=300, bbox_inches='tight')
print("Saved: fig1_ndcg_comparison.png")
plt.close()


# --- Figure 2: Per-group performance distribution (box plot) ---
fig, ax = plt.subplots(figsize=(10, 5))
data_for_box = []
labels_for_box = []
for r in sorted(all_results, key=lambda x: -x['ndcg@10']):
    data_for_box.append(r['ndcg10_values'])
    labels_for_box.append(r['method'])
bp = ax.boxplot(data_for_box, labels=labels_for_box, patch_artist=True, vert=True)
for patch, label in zip(bp['boxes'], labels_for_box):
    if 'Resonant' in label and 'no tower' not in label:
        patch.set_facecolor('#2ecc71')
    else:
        patch.set_facecolor('#bdc3c7')
    patch.set_alpha(0.7)
ax.set_ylabel('NDCG@10', fontsize=12)
ax.set_title('Per-Group Performance Variance', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig2_variance.png'),
            dpi=300, bbox_inches='tight')
print("Saved: fig2_variance.png")
plt.close()


# --- Figure 3: Multi-metric comparison ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_list = ['ndcg@10', 'kendall_tau', 'spearman']
titles  = ['NDCG@10', 'Kendall \u03c4', 'Spearman \u03c1']
for ax, metric, title in zip(axes, metrics_list, titles):
    sorted_r = sorted(all_results, key=lambda x: x[metric])
    vals = [r[metric] for r in sorted_r]
    names = [r['method'] for r in sorted_r]
    colors = ['#2ecc71' if 'Resonant' in n and 'no tower' not in n
              else '#95a5a6' for n in names]
    ax.barh(names, vals, color=colors, alpha=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    if min(vals) < 0:
        ax.axvline(0, color='red', linewidth=0.5, alpha=0.5)
plt.suptitle('All Metrics Comparison (v3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig3_all_metrics.png'),
            dpi=300, bbox_inches='tight')
print("Saved: fig3_all_metrics.png")
plt.close()


# --- Figure 4: Learned factor weights ---
fig, ax = plt.subplots(figsize=(8, 4))
colors_fw = ['#e74c3c' if v in TOKEN_VALUES else '#3498db' for v in NEWS_VALUES]
ax.barh(NEWS_VALUES, fw, color=colors_fw, alpha=0.8, edgecolor='black')
ax.axvline(1.0 / len(NEWS_VALUES), color='gray', linewidth=1, linestyle='--',
           alpha=0.5, label=f'Uniform (1/{len(NEWS_VALUES)})')
ax.set_xlabel('Weight (softmax)', fontsize=11)
ax.set_title('Learned News Value Weights (v3)', fontsize=13, fontweight='bold')
ax.legend(handles=[
    plt.Rectangle((0, 0), 1, 1, color='#e74c3c', alpha=0.8, label='Token-level'),
    plt.Rectangle((0, 0), 1, 1, color='#3498db', alpha=0.8, label='Sentence-level'),
    plt.Line2D([0], [0], color='gray', linestyle='--', label='Uniform'),
], loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig4_factor_weights.png'),
            dpi=300, bbox_inches='tight')
print("Saved: fig4_factor_weights.png")
plt.close()


# --- Figure 5: Predicted vs gold scores (scatter) ---
# Collect predicted scores and gold ranks for scatter plot
all_pred, all_gold = [], []
for group in test_groups:
    headlines = [a["headline"] for a in group]
    bodies    = [a["body"]     for a in group]
    ranks     = [a["rank"]     for a in group]
    scores = sota_score(headlines, bodies)
    all_pred.extend(scores.tolist())
    all_gold.extend(ranks)

all_pred = np.array(all_pred)
all_gold = np.array(all_gold)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ResonantBERT v3
ax = axes[0]
ax.scatter(all_gold, all_pred, alpha=0.15, s=10, c='#2ecc71')
# Trend line: mean prediction per gold rank
for rank_val in sorted(set(all_gold)):
    m = all_gold == rank_val
    ax.plot(rank_val, all_pred[m].mean(), 's', color='darkgreen',
            markersize=10, zorder=5)
ax.set_xlabel('Gold Rank (lower = better)', fontsize=11)
ax.set_ylabel('Predicted Score', fontsize=11)
ax.set_title('ResonantBERT v3', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

# TF-IDF comparison
tfidf_pred, tfidf_gold = [], []
for group in test_groups:
    headlines = [a["headline"] for a in group]
    bodies    = [a["body"]     for a in group]
    ranks     = [a["rank"]     for a in group]
    scores = tfidf_score(headlines, bodies)
    tfidf_pred.extend(scores.tolist())
    tfidf_gold.extend(ranks)
tfidf_pred = np.array(tfidf_pred)
tfidf_gold = np.array(tfidf_gold)

ax = axes[1]
ax.scatter(tfidf_gold, tfidf_pred, alpha=0.15, s=10, c='#95a5a6')
for rank_val in sorted(set(tfidf_gold)):
    m = tfidf_gold == rank_val
    ax.plot(rank_val, tfidf_pred[m].mean(), 's', color='black',
            markersize=10, zorder=5)
ax.set_xlabel('Gold Rank (lower = better)', fontsize=11)
ax.set_ylabel('Predicted Score', fontsize=11)
ax.set_title('TF-IDF + Ridge', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

plt.suptitle('Predicted Score vs. Gold Rank', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig5_pred_vs_gold.png'),
            dpi=300, bbox_inches='tight')
print("Saved: fig5_pred_vs_gold.png")
plt.close()


print(f"\nAll figures saved to: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# EXPORT: LaTeX Tables + JSON
# ============================================================

# LaTeX Table 1 — Main results
latex = "\\begin{table}[t]\n\\centering\n\\small\n"
latex += "\\begin{tabular}{lcccc}\n\\toprule\n"
latex += ("\\textbf{Model} & \\textbf{NDCG@10} & \\textbf{Kendall $\\tau$} "
          "& \\textbf{Spearman $\\rho$} & \\textbf{Pairwise Acc.} \\\\\n")
latex += "\\midrule\n"

for r in sorted(all_results, key=lambda x: -x['ndcg@10']):
    is_sota = ('Resonant' in r['method'] and 'no tower' not in r['method'])
    bold = "\\textbf{" if is_sota else ""
    end_bold = "}" if is_sota else ""
    latex += f"{bold}{r['method']}{end_bold} & "
    latex += f"{r['ndcg@10']:.3f} $\\pm$ {r['ndcg@10_std']:.3f} & "
    latex += f"{r['kendall_tau']:.3f} $\\pm$ {r['kendall_tau_std']:.3f} & "
    latex += f"{r['spearman']:.3f} $\\pm$ {r['spearman_std']:.3f} & "
    latex += f"{r['pairwise_acc']:.3f} \\\\\n"

latex += "\\bottomrule\n\\end{tabular}\n"
latex += ("\\caption{Cross-publisher zero-shot news value ranking results. "
          "ResonantBERT v3 trained on BBC Track~A labels, evaluated on "
          "ranking.json test set.}\n")
latex += "\\label{tab:main}\n\\end{table}"

with open(os.path.join(OUTPUT_DIR, "table1_main.tex"), "w") as f:
    f.write(latex)
print(f"Saved: table1_main.tex")

# LaTeX Table 2 — Factor weights
latex2 = "\\begin{table}[t]\n\\centering\n\\small\n"
latex2 += "\\begin{tabular}{lcc}\n\\toprule\n"
latex2 += "\\textbf{News Value} & \\textbf{Head Type} & \\textbf{Weight} \\\\\n"
latex2 += "\\midrule\n"
for nv, w in zip(NEWS_VALUES, fw):
    ht = "Token" if nv in TOKEN_VALUES else "Sentence"
    latex2 += f"{nv.replace('_', ' ').title()} & {ht} & {w:.4f} \\\\\n"
latex2 += "\\bottomrule\n\\end{tabular}\n"
latex2 += ("\\caption{Learned news value weights after training. "
           "Weights are softmax-normalized.}\n")
latex2 += "\\label{tab:weights}\n\\end{table}"

with open(os.path.join(OUTPUT_DIR, "table2_weights.tex"), "w") as f:
    f.write(latex2)
print(f"Saved: table2_weights.tex")

# JSON export
results_export = {}
for r in all_results:
    results_export[r['method']] = {
        k: float(v) if isinstance(v, (np.floating, float)) else v
        for k, v in r.items()
        if k not in ['per_group', 'ndcg10_values', 'spearman_values']
    }
results_export['factor_weights'] = {
    nv: float(w) for nv, w in zip(NEWS_VALUES, fw)
}

with open(os.path.join(OUTPUT_DIR, "results_v3.json"), "w") as f:
    json.dump(results_export, f, indent=2, default=str)
print(f"Saved: results_v3.json")

# Summary
print("\n" + "=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"Files: {sorted(os.listdir(OUTPUT_DIR))}")
print(f"\nHeadline: ResonantBERT v3 achieves "
      f"{results_sota['ndcg@10']:.4f} NDCG@10, "
      f"{results_sota['spearman']:.4f} Spearman rho")

best_baseline = max(
    [r for r in all_results
     if 'Resonant' not in r['method'] and 'Random' not in r['method']],
    key=lambda x: x['spearman'])

delta_rho = results_sota['spearman'] - best_baseline['spearman']
print(f"vs. best baseline ({best_baseline['method']}): "
      f"Spearman {delta_rho:+.4f}")
print(f"\nDone.")